# ![](https://ga-dash.s3.amazonaws.com/production/assets/logo-9f88ae6c9c3871690e33280fcf557f33.png) Lesson 3.03 | Feature Engineering

## Review of Linear Regression

- Linear regression is a way for us to relate some dependent variable $Y$ to independent variables $X_1$,$\ldots$,$X_p$.
- We might write this out in one of the following two forms:
$$
\begin{eqnarray}
Y &=& \beta_0 + \beta_1X_1 + \beta_2X_2 + \cdots + \beta_pX_p +\varepsilon\\
\mathbf{Y} &=& \mathbf{X \beta + \varepsilon}
\end{eqnarray}
$$

<details><summary>
There are four assumptions to the simple linear regression model and five assumptions to the multiple linear regression model.
</summary>
1. **Linearity:** $Y$ is linearly related to $X_i$ for all $i$.
2. **Independence:** Each residual $\varepsilon_i$ is independent of $\varepsilon_j$ for all $i\neq j$.
3. **Normality:** The errors (residuals) follow a Normal distribution with mean 0.
4. **Equality of Variance:** The errors (residuals) should have a roughly consistent pattern, regardless of the value of $X_i$. (There should be no discernable relationship between $X_i$ and the residuals.)
5. **Independence Part II:** $X_i$ is independent of $X_j$ for all $i\neq j$.
</details>
- We can measure the performance of our model by using mean squared error (MSE).

## Feature Engineering

- If I use degrees Fahrenheit to predict how much a substance will expand or inches of rain to predict traffic accidents, people outside the United States may have a tougher time understanding my work.
- If I use straight line distance (as the crow flies) between two locations, my estimated time of arrival in a taxi or a Lyft is going to be pretty bad.
- If I put text into my model without some sort of preprocessing, my computer isn't going to understand how to handle it.

Suffice it to say: If your features (variables) aren't good, your predictions and inferences won't be good!

#### What is feature engineering?

"Coming up with features is difficult, time-consuming, requires expert knowledge. 'Applied machine learning' is basically feature engineering." - Andrew Ng

**Feature engineering** is the term broadly applied to the creation and manipulation of features (variables) used in machine learning algorithms.

Unless we're working with the same data over and over again, this isn't something we can automate. It will require creativity and a good, thorough understanding of our data.

#### The Process of Data Science
1. Data Gathering
2. Data Cleaning/Munging
3. EDA
4. Modeling
5. Reporting
    - Feature engineering will straddle all five of these steps, but mostly focus on steps 2 and 3.

#### [The Process of Feature Engineering](https://www.youtube.com/watch?v=drUToKxEAUA)
1. Brainstorming or testing features.
2. Deciding what features to create.
3. Creating features.
4. Checking how the features work with your model.
5. Improving features (if needed).
6. Return to step 1.
7. "Do data science!"

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("forestfires.csv")

In [ ]:
df.head()

[Documentation](https://archive.ics.uci.edu/ml/machine-learning-databases/forest-fires/forestfires.names)

1. Number of Instances: 517 

2. Number of Attributes: 12 + output attribute
  
   Note: several of the attributes may be correlated, thus it makes sense to apply some sort of feature selection.

3. Attribute information:

   For more information, read [Cortez and Morais, 2007].

   - X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
   - Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
   - month - month of the year: "jan" to "dec" 
   - day - day of the week: "mon" to "sun"
   - FFMC - FFMC index from the FWI system: 18.7 to 96.20
   - DMC - DMC index from the FWI system: 1.1 to 291.3 
   - DC - DC index from the FWI system: 7.9 to 860.6 
   - ISI - ISI index from the FWI system: 0.0 to 56.10
   - temp - temperature in Celsius degrees: 2.2 to 33.30
   - RH - relative humidity in %: 15.0 to 100
   - wind - wind speed in km/h: 0.40 to 9.40 
   - rain - outside rain in mm/m2 : 0.0 to 6.4 
   - area - the burned area of the forest (in ha): 0.00 to 1090.84 (this output variable is very skewed towards 0.0, thus it may make sense to model with the logarithm transform). 

In [ ]:
import statsmodels.api as sm

What do each of the next six cells do? Chat in your local markets and we'll summarize together.

In [ ]:
dep = df['area'] # Cell 1

In [ ]:
indep = df.drop(['area','month','day'], axis = 'columns') # Cell 2

In [ ]:
indep = sm.add_constant(indep) # Cell 3

In [ ]:
model = sm.OLS(dep,indep) # Cell 4

In [ ]:
results = model.fit() # Cell 5

In [ ]:
results.summary() # Cell 6

Now, let's walk through some feature engineering!

#### Conversion

First, let's say that we want to convert `temp` from Celsius to Fahrenheit. Let's do that now by engineering a new feature `Fahr`, then build a new model and check the summary.

This is an example of a conversion. In this case, our goal isn't to add any new information to the model - we haven't given our model any "new" information - we just want to change the way our model accounts for temperature by using Fahrenheit!

**Check:** How would I interpret the coefficient for `temp` here? (Remember that our `Y` variable is `area`.)

#### Dummy Variables
Second, let's say that we want to convert `day` into a dummy variable. (A variable with values of `1`s and `0`s; enables our computer to understand categorical data.) There are a few ways of doing this. We'll go through one of the most common, `get_dummies()`.

In [ ]:
df.head()

**Check:** How would I interpret the coefficient for `Mon`?

#### Advanced Feature Engineering
There's not an exhaustive list of ways we can engineer features. However, let's chat about a use-case in the context of this data.

We have horizontal and vertical spatial coordinates, listed as `X` and `Y` in our data. Perhaps we believe that area (our dependent variable) is linearly related to `X` or `Y`, but that's not very likely.

What if we think that `X` and `Y` have some combined effect on area? Here, we can create an **[interaction term](http://statisticsbyjim.com/regression/interaction-effects/)**.
- An interaction term is a way for us to account for the case when two independent variables have some joint effect or interact in some way.

Interpreting interaction terms is much less straightforward than interpreting "marginal" terms.

---

The last feature we'll engineer has to do with distance.

We have geographic data across a grid. We can talk about how horizontal distance matters or how vertical distance matters. We can also talk about how horizontal and vertical interact, but this is still clunky and awkward for us to handle.

Instead, what if we could actually define **distance** from a certain point?

$$
dist(A,B) = \sqrt{(A_X - B_X) ^ 2 + (A_Y - B_Y) ^ 2}
$$

Let's say that there's a specific landmark of interest at the point $X = 6, Y = 5$, and we want to engineer a new feature that represents distance from this landmark.

## Conclusion

#### When do we decide when to stop?
- There's never infinite time.
- We have to consider **both** a) how useful it is and b) how much time you're willing to invest.
    - You will **always** be able to engineer more features.
    - [Diminishing marginal returns](https://www.investopedia.com/terms/l/lawofdiminishingmarginalreturn.asp) will absolutely play a role here.